# Master's project: constrained customer segmentation
## Research questions
1. How does standard K-means compare statistically with RFM-based K-means and constrained K-means?
2. Does constrained clustering improve separation in future business metrics relative to unconstrained clustering?

**Design:** Use an earlier transaction window to form clusters and a later 90-day window to evaluate future revenue, orders, average order value, units and product diversity. This reduces circularity from evaluating clusters on the same spending data used to create them.

**Constraint caveat:** UCI Online Retail has no verified customer-segment labels. The notebook creates weak must-link/cannot-link proxy constraints from customers' modal country in the training period. These are not ground truth and may encode geographic patterns. Treat this as a methodological experiment, not proof of business ROI.

## 1. Install and import

In [ ]:
%pip -q install openpyxl scikit-learn scipy seaborn

In [ ]:
import os, urllib.request, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## 2. Download and load the UCI Online Retail dataset

In [ ]:
FILE_PATH = "/content/online_retail.xlsx"
URLS = [
 "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx",
 "https://archive.ics.uci.edu/static/public/352/online+retail.xlsx"
]
if not os.path.exists(FILE_PATH):
    for url in URLS:
        try:
            req = urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                content = r.read()
            if len(content) < 100_000: raise ValueError("Unexpectedly small download.")
            open(FILE_PATH, "wb").write(content)
            pd.read_excel(FILE_PATH, engine="openpyxl", nrows=3)
            print("Downloaded:", url)
            break
        except Exception as e:
            print("Download failed:", repr(e))
            if os.path.exists(FILE_PATH): os.remove(FILE_PATH)
if not os.path.exists(FILE_PATH):
    from google.colab import files
    print("Download Online Retail.xlsx from https://archive.ics.uci.edu/dataset/352/online+retail and upload it.")
    uploaded = files.upload()
    candidates = [x for x in uploaded if x.lower().endswith(".xlsx")]
    if not candidates: raise FileNotFoundError("Please upload Online Retail.xlsx.")
    FILE_PATH = "/content/" + candidates[0]
raw = pd.read_excel(FILE_PATH, engine="openpyxl")
raw.columns = raw.columns.astype(str).str.strip().str.replace(" ", "_")
print("Raw shape:", raw.shape)
display(raw.head())

## 3. Clean data and create a temporal holdout

In [ ]:
df = raw.copy()
df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df["Country"] = df["Country"].astype("string").str.strip()
df = df.dropna(subset=["InvoiceNo","InvoiceDate","Quantity","UnitPrice","CustomerID","StockCode","Country"]).copy()
df = df[(~df["InvoiceNo"].str.upper().str.startswith("C")) & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
df["CustomerID"] = df["CustomerID"].astype("int64")
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
df["InvoiceDay"] = df["InvoiceDate"].dt.normalize()
test_start = df["InvoiceDate"].max().normalize() - pd.Timedelta(days=90)
train_tx = df[df["InvoiceDate"] < test_start].copy()
future_tx = df[df["InvoiceDate"] >= test_start].copy()
print("Training:", train_tx["InvoiceDate"].min(), "to", train_tx["InvoiceDate"].max(), "| rows:", len(train_tx))
print("Future holdout:", future_tx["InvoiceDate"].min(), "to", future_tx["InvoiceDate"].max(), "| rows:", len(future_tx))

## 4. Engineer training features and future business outcomes

In [ ]:
def mean_gap(dates):
    days = pd.Series(pd.to_datetime(dates).dt.normalize().unique()).sort_values()
    return days.diff().dropna().dt.days.mean() if len(days) >= 2 else np.nan

snapshot = train_tx["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
customers = train_tx.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate","max"), FirstPurchase=("InvoiceDate","min"),
    Frequency=("InvoiceNo","nunique"), Monetary=("Revenue","sum"),
    TotalUnits=("Quantity","sum"), ProductDiversity=("StockCode","nunique")
).reset_index()
customers["Recency"] = (snapshot - customers["LastPurchase"].dt.normalize()).dt.days
customers["CustomerTenureDays"] = (customers["LastPurchase"] - customers["FirstPurchase"]).dt.days
train_orders = train_tx.groupby(["CustomerID","InvoiceNo"]).agg(OrderRevenue=("Revenue","sum")).reset_index()
customers["AverageOrderValue"] = customers["CustomerID"].map(train_orders.groupby("CustomerID")["OrderRevenue"].mean())
customers["MeanDaysBetweenPurchases"] = customers["CustomerID"].map(train_tx.groupby("CustomerID")["InvoiceDate"].apply(mean_gap))
country = train_tx.groupby("CustomerID")["Country"].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
customers["Country"] = customers["CustomerID"].map(country)

future = future_tx.groupby("CustomerID").agg(
    FutureRevenue=("Revenue","sum"), FutureOrders=("InvoiceNo","nunique"),
    FutureUnits=("Quantity","sum"), FutureProductDiversity=("StockCode","nunique")
).reset_index()
customers = customers.merge(future, on="CustomerID", how="left")
future_cols = ["FutureRevenue","FutureOrders","FutureUnits","FutureProductDiversity"]
customers[future_cols] = customers[future_cols].fillna(0)
future_orders = future_tx.groupby(["CustomerID","InvoiceNo"]).agg(OrderRevenue=("Revenue","sum")).reset_index()
future_aov = future_orders.groupby("CustomerID")["OrderRevenue"].mean()
customers["FutureAverageOrderValue"] = customers["CustomerID"].map(future_aov).fillna(0)
print("Customers used:", len(customers), "| future purchasers:", int((customers["FutureOrders"] > 0).sum()))
display(customers.head())

## 5. Transform and scale features

In [ ]:
RFM_FEATURES = ["Recency","Frequency","Monetary"]
EXTENDED_FEATURES = ["Recency","Frequency","Monetary","AverageOrderValue","TotalUnits","ProductDiversity","MeanDaysBetweenPurchases"]
def prepare_matrix(data, cols):
    x = data[cols].copy()
    for col in cols:
        x[col] = pd.to_numeric(x[col], errors="coerce").replace([np.inf,-np.inf], np.nan)
        x[col] = x[col].fillna(x[col].median()).clip(lower=0)
        x[col] = np.log1p(x[col])
    scaler = RobustScaler()
    return scaler.fit_transform(x)
X_rfm = prepare_matrix(customers, RFM_FEATURES)
X_ext = prepare_matrix(customers, EXTENDED_FEATURES)
print(X_rfm.shape, X_ext.shape)

## 6. Generate sampled country proxy constraints
These pairs use training-period country only, not future business outcomes. They are weak proxy constraints rather than verified segment labels.

In [ ]:
def generate_constraints(data, n_must=150, n_cannot=150, seed=42, max_draws=250000):
    rng = np.random.default_rng(seed)
    countries = data["Country"].astype(str).to_numpy()
    n = len(data); must=set(); cannot=set(); draws=0
    while (len(must)<n_must or len(cannot)<n_cannot) and draws<max_draws:
        i,j = rng.integers(0,n,size=2); draws += 1
        if i==j: continue
        pair=(int(min(i,j)),int(max(i,j)))
        if countries[i] == countries[j] and len(must)<n_must: must.add(pair)
        elif countries[i] != countries[j] and len(cannot)<n_cannot: cannot.add(pair)
    return sorted(must), sorted(cannot)
must_link, cannot_link = generate_constraints(customers, seed=RANDOM_STATE)
print("Must-link:",len(must_link),"Cannot-link:",len(cannot_link),"Countries:",customers["Country"].nunique())

## 7. Implement soft constrained K-means
This heuristic penalises violated pairwise constraints. It does not guarantee every constraint will be satisfied, so violation rates are reported.

In [ ]:
class SoftConstrainedKMeans:
    def __init__(self, n_clusters=4, constraint_weight=1.0, max_iter=35, n_init=4, random_state=42):
        self.k=n_clusters; self.weight=constraint_weight; self.max_iter=max_iter
        self.n_init=n_init; self.random_state=random_state
    def fit_predict(self, X, must_link, cannot_link):
        X=np.asarray(X,float); best_obj=np.inf; best=None
        master=np.random.default_rng(self.random_state); n=len(X)
        ml_adj=[[] for _ in range(n)]; cl_adj=[[] for _ in range(n)]
        for i,j in must_link: ml_adj[i].append(j); ml_adj[j].append(i)
        for i,j in cannot_link: cl_adj[i].append(j); cl_adj[j].append(i)
        for init in range(self.n_init):
            seed=int(master.integers(0,2**31-1)); rng=np.random.default_rng(seed)
            labels=KMeans(n_clusters=self.k,n_init=1,random_state=seed).fit_predict(X)
            for iteration in range(self.max_iter):
                old=labels.copy()
                for i in rng.permutation(n):
                    centroids=np.vstack([X[labels==c].mean(axis=0) if np.any(labels==c) else X[rng.integers(n)] for c in range(self.k)])
                    costs=((centroids-X[i])**2).sum(axis=1)
                    for c in range(self.k):
                        costs[c] += self.weight * sum(labels[j]!=c for j in ml_adj[i])
                        costs[c] += self.weight * sum(labels[j]==c for j in cl_adj[i])
                    labels[i]=int(np.argmin(costs))
                for c in range(self.k):
                    if not np.any(labels==c):
                        counts=np.bincount(labels,minlength=self.k)
                        centroids=np.vstack([X[labels==j].mean(axis=0) if np.any(labels==j) else X[0] for j in range(self.k)])
                        errors=((X-centroids[labels])**2).sum(axis=1)
                        for idx in np.argsort(errors)[::-1]:
                            if counts[labels[idx]]>1:
                                labels[idx]=c; break
                if np.array_equal(old,labels): break
            centroids=np.vstack([X[labels==c].mean(axis=0) for c in range(self.k)])
            inertia=((X-centroids[labels])**2).sum()
            violations=sum(labels[i]!=labels[j] for i,j in must_link)+sum(labels[i]==labels[j] for i,j in cannot_link)
            objective=inertia+self.weight*violations
            if objective<best_obj: best_obj=objective; best=labels.copy()
        self.labels_=best; self.objective_=best_obj
        return best

def violation_rates(labels, ml, cl):
    return (np.mean([labels[i]!=labels[j] for i,j in ml]) if ml else np.nan,
            np.mean([labels[i]==labels[j] for i,j in cl]) if cl else np.nan)
print("Constrained clustering class ready.")

## 8. Fit three methods with the same initial k

In [ ]:
K=4
labels_standard=KMeans(n_clusters=K,n_init=20,random_state=RANDOM_STATE).fit_predict(X_ext)
labels_rfm=KMeans(n_clusters=K,n_init=20,random_state=RANDOM_STATE).fit_predict(X_rfm)
weight=1.0
constrained=SoftConstrainedKMeans(n_clusters=K,constraint_weight=weight,random_state=RANDOM_STATE)
labels_constrained=constrained.fit_predict(X_ext,must_link,cannot_link)
assignments={"Standard K-means":labels_standard,"RFM K-means":labels_rfm,"Constrained K-means":labels_constrained}
for name,labels in assignments.items(): print(name, pd.Series(labels).value_counts().sort_index().to_dict())
print("Must-link / cannot-link violation rates:", violation_rates(labels_constrained,must_link,cannot_link))

## 9. Compare statistical clustering metrics

In [ ]:
def internal_metrics(X, labels):
    counts=pd.Series(labels).value_counts()
    out={"Clusters":len(counts),"Silhouette":np.nan,"DaviesBouldin":np.nan,"CalinskiHarabasz":np.nan,
         "MinClusterSize":int(counts.min()),"MaxClusterSize":int(counts.max())}
    if len(counts)>1 and len(labels)>len(counts):
        out["Silhouette"]=silhouette_score(X,labels,sample_size=min(10000,len(X)),random_state=RANDOM_STATE)
        out["DaviesBouldin"]=davies_bouldin_score(X,labels)
        out["CalinskiHarabasz"]=calinski_harabasz_score(X,labels)
    return out
stat_rows=[]
for name,labels in assignments.items():
    x=X_rfm if name=="RFM K-means" else X_ext
    stat_rows.append({"Method":name,**internal_metrics(x,labels)})
stat_results=pd.DataFrame(stat_rows)
display(stat_results)

## 10. Evaluate separation in future business outcomes
Eta-squared measures the share of variation in a future metric associated with cluster membership. It is descriptive, not causal.

In [ ]:
FUTURE_METRICS=["FutureRevenue","FutureOrders","FutureAverageOrderValue","FutureUnits","FutureProductDiversity"]
def eta_squared(values,labels):
    d=pd.DataFrame({"y":np.asarray(values,dtype=float),"cluster":np.asarray(labels)})
    total=((d.y-d.y.mean())**2).sum()
    if total==0: return np.nan
    between=sum(len(g)*(g.y.mean()-d.y.mean())**2 for _,g in d.groupby("cluster"))
    return between/total
rows=[]
for method,labels in assignments.items():
    for metric in FUTURE_METRICS:
        rows.append({"Method":method,"FutureMetric":metric,"EtaSquared":eta_squared(customers[metric],labels)})
business_separation=pd.DataFrame(rows)
display(business_separation.pivot(index="FutureMetric",columns="Method",values="EtaSquared").round(4))

## 11. Profile future business outcomes by segment

In [ ]:
def future_profile(data,labels,method):
    d=data.copy(); d["Cluster"]=labels
    p=d.groupby("Cluster").agg(
        Customers=("CustomerID","nunique"),
        FutureRevenueMean=("FutureRevenue","mean"),
        FutureRevenueMedian=("FutureRevenue","median"),
        FutureOrdersMean=("FutureOrders","mean"),
        FutureAOVMean=("FutureAverageOrderValue","mean"),
        FutureUnitsMean=("FutureUnits","mean"),
        FutureProductDiversityMean=("FutureProductDiversity","mean")
    ).reset_index()
    p["CustomerShare"]=p["Customers"]/len(d)
    totals=d.groupby("Cluster")["FutureRevenue"].sum()
    p["FutureRevenueShare"]=p["Cluster"].map(totals)/max(d["FutureRevenue"].sum(),1e-12)
    p["Method"]=method
    return p
profiles=pd.concat([future_profile(customers,l,m) for m,l in assignments.items()],ignore_index=True)
display(profiles.sort_values(["Method","Cluster"]))

## 12. Constraint weight sensitivity
Do not choose the weight solely because it maximises business separation; examine the trade-off among statistical quality, constraint violations and future outcomes.

In [ ]:
sensitivity_rows=[]
for w in [0.1,0.5,1.0,2.0,5.0]:
    model=SoftConstrainedKMeans(n_clusters=K,constraint_weight=w,n_init=3,max_iter=30,random_state=RANDOM_STATE)
    lab=model.fit_predict(X_ext,must_link,cannot_link)
    mlr,clr=violation_rates(lab,must_link,cannot_link)
    sensitivity_rows.append({"Weight":w,**internal_metrics(X_ext,lab),
        "MustLinkViolation":mlr,"CannotLinkViolation":clr,
        "FutureRevenueEta2":eta_squared(customers["FutureRevenue"],lab),
        "FutureOrdersEta2":eta_squared(customers["FutureOrders"],lab),
        "FutureAOVEta2":eta_squared(customers["FutureAverageOrderValue"],lab)})
sensitivity=pd.DataFrame(sensitivity_rows)
display(sensitivity)

## 13. Robustness to different sampled constraint sets

In [ ]:
robust_rows=[]
for seed in [11,22,33,44,55]:
    ml,cl=generate_constraints(customers,seed=seed)
    model=SoftConstrainedKMeans(n_clusters=K,constraint_weight=1.0,n_init=3,max_iter=30,random_state=seed)
    lab=model.fit_predict(X_ext,ml,cl)
    mlr,clr=violation_rates(lab,ml,cl)
    robust_rows.append({"Seed":seed,"Silhouette":silhouette_score(X_ext,lab,sample_size=min(10000,len(X_ext)),random_state=RANDOM_STATE),
        "FutureRevenueEta2":eta_squared(customers["FutureRevenue"],lab),
        "FutureOrdersEta2":eta_squared(customers["FutureOrders"],lab),
        "FutureAOVEta2":eta_squared(customers["FutureAverageOrderValue"],lab),
        "MustLinkViolation":mlr,"CannotLinkViolation":clr})
robustness=pd.DataFrame(robust_rows)
display(robustness)
display(robustness.drop(columns="Seed").agg(["mean","std"]).T)

## 14. Visualise assignments with PCA

In [ ]:
coords=PCA(n_components=2,random_state=RANDOM_STATE).fit_transform(X_ext)
plotdf=pd.DataFrame({"PC1":coords[:,0],"PC2":coords[:,1],"Standard":labels_standard.astype(str),"Constrained":labels_constrained.astype(str)})
fig,axes=plt.subplots(1,2,figsize=(14,5))
sns.scatterplot(data=plotdf,x="PC1",y="PC2",hue="Standard",alpha=.6,s=22,linewidth=0,ax=axes[0]); axes[0].set_title("Standard K-means")
sns.scatterplot(data=plotdf,x="PC1",y="PC2",hue="Constrained",alpha=.6,s=22,linewidth=0,ax=axes[1]); axes[1].set_title("Constrained K-means")
plt.tight_layout(); plt.show()

## 15. Export outputs

In [ ]:
OUT="/content/constrained_customer_segmentation_results"
os.makedirs(OUT,exist_ok=True)
customers.to_csv(f"{OUT}/customer_features_and_future_outcomes.csv",index=False)
stat_results.to_csv(f"{OUT}/statistical_validation.csv",index=False)
business_separation.to_csv(f"{OUT}/future_business_separation.csv",index=False)
profiles.to_csv(f"{OUT}/future_business_segment_profiles.csv",index=False)
sensitivity.to_csv(f"{OUT}/constraint_weight_sensitivity.csv",index=False)
robustness.to_csv(f"{OUT}/constraint_set_robustness.csv",index=False)
print("Saved outputs in",OUT)
print("\n".join(sorted(os.listdir(OUT))))

## 16. Dissertation interpretation checklist
- **RQ1:** Compare silhouette (higher usually better), Davies–Bouldin (lower usually better), Calinski–Harabasz (higher usually better), cluster sizes and robustness. RFM and extended feature spaces differ, so interpret their internal metrics cautiously.
- **RQ2:** Compare eta-squared for future revenue, orders, AOV, units and product diversity. Report the absolute change from standard K-means to constrained K-means.
- Report constraint violation rates and sensitivity to constraint weights and sampled pairs.
- Country constraints are weak proxies and may correlate with future behaviour; they are not true labels.
- Greater business-metric separation is not proof of increased profit, ROI, or causal campaign effectiveness.
- Consider using independently elicited business constraints if a company or domain expert can provide them.